# Deep learning for computer vision


This notebook will teach you to build and train convolutional networks for image recognition. Brace yourselves. Thanks [Yandex Data School Analysis](https://github.com/yandexdataschool/Practical_DL/blob/fall21/week03_convnets/seminar_pytorch.ipynb) for this work!

# CIFAR dataset
This week, we shall focus on the image recognition problem on cifar10 dataset
* 60k images of shape 3x32x32
* 10 different classes: planes, dogs, cats, trucks, etc.

![image](https://github.com/yandexdataschool/Practical_DL/raw/fall21/week03_convnets/cifar10.jpg)

In [1]:
# when running in colab, un-comment this
!wget https://raw.githubusercontent.com/yandexdataschool/Practical_DL/fall19/week03_convnets/cifar.py

--2026-05-03 13:50:20--  https://raw.githubusercontent.com/yandexdataschool/Practical_DL/fall19/week03_convnets/cifar.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2396 (2.3K) [text/plain]
Saving to: ‘cifar.py’

cifar.py            100%[===================>]   2.34K  --.-KB/s    in 0s      

2026-05-03 13:50:20 (44.1 MB/s) - ‘cifar.py’ saved [2396/2396]



In [14]:
# import numpy as np
# from cifar import load_cifar10
# X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10("/content/cifar_data")

# class_names = np.array(['airplane', 'automobile', 'bird', 'cat', 'deer',
#                         'dog', 'frog', 'horse', 'ship', 'truck'])

# print(X_train.shape,y_train.shape)

Dataset not found. Downloading...


HTTPError: HTTP Error 503: Service Unavailable

# У меня не грузиться датасет, поэтому пробую из torchvision

In [3]:
# UToronto не отвечает — грузим данные с Kaggle
# Нужен API-токен: kaggle.com → Settings → API → Create New Token → скачать kaggle.json
!pip install -q kaggle

from google.colab import files
uploaded = files.upload()   # загрузить kaggle.json

import os, shutil
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

os.makedirs("./cifar_data", exist_ok=True)
!kaggle competitions download -c cifar-10 -p ./cifar_data/
!cd cifar_data && unzip -q -o cifar-10.zip
!apt-get install -q -y p7zip-full
!7z x ./cifar_data/train.7z -o./cifar_data/ -y > /dev/null
print("Данные скачаны и распакованы!")


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split

labels_df = pd.read_csv("./cifar_data/trainLabels.csv")

class_names = np.array(["airplane", "automobile", "bird", "cat", "deer",
                        "dog", "frog", "horse", "ship", "truck"])
class_to_idx = {c: i for i, c in enumerate(class_names)}

def load_img(img_id):
    return np.array(Image.open(f"./cifar_data/train/{img_id}.png"))

print("Загружаем 50k изображений, ~1-2 мин...")
with ThreadPoolExecutor(max_workers=4) as executor:
    images = list(executor.map(load_img, labels_df["id"].values))
print("Загружено")

# uint8 (N, H, W, C) — для PIL-transforms
X_all_raw = np.array(images)
# float32 (N, C, H, W) в [0,1] — для numpy-based циклов
X_all = X_all_raw.transpose(0, 3, 1, 2).astype(np.float32) / 255.0
y_all = np.array([class_to_idx[l] for l in labels_df["label"]])

# split 80/10/10 (ground truth для competition test нет — делаем своё разбиение)
idx = np.arange(len(y_all))
idx_tv, idx_test   = train_test_split(idx, test_size=5000, random_state=42)
idx_train, idx_val = train_test_split(idx_tv, test_size=5000, random_state=42)

X_train_raw = X_all_raw[idx_train]
X_val_raw   = X_all_raw[idx_val]
X_test_raw  = X_all_raw[idx_test]
X_train = X_all[idx_train]
X_val   = X_all[idx_val]
X_test  = X_all[idx_test]
y_train = y_all[idx_train]
y_val   = y_all[idx_val]
y_test  = y_all[idx_test]

print(X_train.shape, y_train.shape)


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=[12,10])
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.xlabel(class_names[y_train[i]])
    plt.imshow(np.transpose(X_train[i],[1,2,0]))

# Building a network

Simple neural networks with layers applied on top of one another can be implemented as `torch.nn.Sequential` - just add a list of pre-built modules and let it train.

In [ ]:
import torch, torch.nn as nn
import torch.nn.functional as F

# a special module that converts [batch, channel, w, h] to [batch, units]
class Flatten(nn.Module):
    def forward(self, input):
        return input.view(input.size(0), -1)

Let's start with a dense network for our baseline:

In [ ]:
model = nn.Sequential()

# reshape from "images" to flat vectors
model.add_module('flatten', Flatten())

# dense "head"
model.add_module('dense1', nn.Linear(3 * 32 * 32, 64))
model.add_module('dense1_relu', nn.ReLU())
model.add_module('dense2_logits', nn.Linear(64, 10)) # logits for 10 classes

As in our basic tutorial, we train our model with negative log-likelihood aka crossentropy.

In [ ]:
def compute_loss(X_batch, y_batch):
    X_batch = torch.as_tensor(X_batch, dtype=torch.float32)
    y_batch = torch.as_tensor(y_batch, dtype=torch.int64)
    logits = model(X_batch)
    return F.cross_entropy(logits, y_batch).mean()

In [ ]:
# example
compute_loss(X_train[:5], y_train[:5])

### Training on minibatches
* We got 40k images, that's way too many for a full-batch SGD. Let's train on minibatches instead
* Below is a function that splits the training sample into minibatches

In [ ]:
# An auxilary function that returns mini-batches for neural network training
def iterate_minibatches(X, y, batchsize):
    indices = np.random.permutation(np.arange(len(X)))
    for start in range(0, len(indices), batchsize):
        ix = indices[start: start + batchsize]
        yield X[ix], y[ix]

In [ ]:
opt = torch.optim.SGD(model.parameters(), lr=0.01)

train_loss = []
val_accuracy = []

In [ ]:
import time
num_epochs = 100 # total amount of full passes over training data
batch_size = 50  # number of samples processed in one SGD iteration

for epoch in range(num_epochs):
    # In each epoch, we do a full pass over the training data:
    start_time = time.time()
    model.train(True) # enable dropout / batch_norm training behavior
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        # train on batch
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.data.numpy())
        
    # And a full pass over the validation data:
    model.train(False) # disable dropout / use averages for batch_norm
    for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
        logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
        y_pred = logits.max(1)[1].data.numpy()
        val_accuracy.append(np.mean(y_batch == y_pred))

    
    # Then we print the results for this epoch:
    print("Epoch {} of {} took {:.3f}s".format(
        epoch + 1, num_epochs, time.time() - start_time))
    print("  training loss (in-iteration): \t{:.6f}".format(
        np.mean(train_loss[-len(X_train) // batch_size :])))
    print("  validation accuracy: \t\t\t{:.2f} %".format(
        np.mean(val_accuracy[-len(X_val) // batch_size :]) * 100))

Don't wait for full 100 epochs. You can interrupt training after 5-20 epochs once validation accuracy stops going up.
```

```

```

```

```

```

```

```

```

```

### Final test

In [ ]:
model.train(False) # disable dropout / use averages for batch_norm
test_batch_acc = []
for X_batch, y_batch in iterate_minibatches(X_test, y_test, 500):
    logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
    y_pred = logits.max(1)[1].data.numpy()
    test_batch_acc.append(np.mean(y_batch == y_pred))

test_accuracy = np.mean(test_batch_acc)
    
print("Final results:")
print("  test accuracy:\t\t{:.2f} %".format(
    test_accuracy * 100))

if test_accuracy * 100 > 95:
    print("Double-check, than consider applying for NIPS'17. SRSly.")
elif test_accuracy * 100 > 90:
    print("U'r freakin' amazin'!")
elif test_accuracy * 100 > 80:
    print("Achievement unlocked: 110lvl Warlock!")
elif test_accuracy * 100 > 70:
    print("Achievement unlocked: 80lvl Warlock!")
elif test_accuracy * 100 > 60:
    print("Achievement unlocked: 70lvl Warlock!")
elif test_accuracy * 100 > 50:
    print("Achievement unlocked: 60lvl Warlock!")
else:
    print("We need more magic! Follow instructons below")

## Task I: small convolution net
### First step

Let's create a mini-convolutional network with roughly such architecture:
* Input layer
* 3x3 convolution with 10 filters and _ReLU_ activation
* 2x2 pooling (or set previous convolution stride to 3)
* Flatten
* Dense layer with 100 neurons and _ReLU_ activation
* 10% dropout
* Output dense layer.


__Convolutional layers__ in torch are just like all other layers, but with a specific set of parameters:

__`...`__

__`model.add_module('conv1', nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3)) # convolution`__

__`model.add_module('pool1', nn.MaxPool2d(2)) # max pooling 2x2`__

__`...`__


Once you're done (and compute_loss no longer raises errors), train it with __Adam__ optimizer with default params (feel free to modify the code above).

If everything is right, you should get at least __50%__ validation accuracy.

In [ ]:
model = nn.Sequential()

model.add_module('conv1', nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3))
model.add_module('relu1', nn.ReLU())
model.add_module('pool1', nn.MaxPool2d(2))
model.add_module('flatten', Flatten())
# conv: 3x32x32 -> 10x30x30, pool: 10x15x15 => 10*15*15 = 2250
model.add_module('dense1', nn.Linear(2250, 100))
model.add_module('relu2', nn.ReLU())
model.add_module('dropout1', nn.Dropout(0.1))
model.add_module('dense2_logits', nn.Linear(100, 10))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss = []
val_accuracy = []

num_epochs = 20
batch_size = 64

for epoch in range(num_epochs):
    start_time = time.time()
    model.train(True)
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.data.numpy())

    model.train(False)
    for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
        logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
        y_pred = logits.max(1)[1].data.numpy()
        val_accuracy.append(np.mean(y_batch == y_pred))

    print("Epoch {} of {} took {:.3f}s".format(epoch + 1, num_epochs, time.time() - start_time))
    print("  training loss: \t{:.6f}".format(np.mean(train_loss[-len(X_train) // batch_size:])))
    print("  validation accuracy: \t{:.2f} %".format(np.mean(val_accuracy[-len(X_val) // batch_size:]) * 100))

In [ ]:
model = nn.Sequential()

model.add_module('conv1', nn.Conv2d(in_channels=3, out_channels=10, kernel_size=3))
model.add_module('bn1', nn.BatchNorm2d(10))
model.add_module('relu1', nn.ReLU())
model.add_module('pool1', nn.MaxPool2d(2))
model.add_module('flatten', Flatten())
model.add_module('dense1', nn.Linear(2250, 100))
model.add_module('bn2', nn.BatchNorm1d(100))
model.add_module('relu2', nn.ReLU())
model.add_module('dropout1', nn.Dropout(0.1))
model.add_module('dense2_logits', nn.Linear(100, 10))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loss = []
val_accuracy = []

for epoch in range(num_epochs):
    start_time = time.time()
    model.train(True)
    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size):
        loss = compute_loss(X_batch, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        train_loss.append(loss.data.numpy())

    model.train(False)
    for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size):
        logits = model(torch.as_tensor(X_batch, dtype=torch.float32))
        y_pred = logits.max(1)[1].data.numpy()
        val_accuracy.append(np.mean(y_batch == y_pred))

    print("Epoch {} of {} took {:.3f}s".format(epoch + 1, num_epochs, time.time() - start_time))
    print("  training loss: \t{:.6f}".format(np.mean(train_loss[-len(X_train) // batch_size:])))
    print("  validation accuracy: \t{:.2f} %".format(np.mean(val_accuracy[-len(X_val) // batch_size:]) * 100))

```

```

```

```

```

```

```

```

```

```

__Hint:__ If you don't want to compute shapes by hand, just plug in any shape (e.g. 1 unit) and run compute_loss. You will see something like this:

__`RuntimeError: size mismatch, m1: [5 x 1960], m2: [1 x 64] at /some/long/path/to/torch/operation`__

See the __1960__ there? That's your actual input shape.

## Task 2: adding normalization

* Add batch norm (with default params) between convolution and ReLU
  * nn.BatchNorm*d (1d for dense, 2d for conv)
  * usually better to put them after linear/conv but before nonlinearity
* Re-train the network with the same optimizer, it should get at least 60% validation accuracy at peak.



## Task 3: Data Augmentation

There's a powerful torch tool for image preprocessing useful to do data preprocessing and augmentation.

Here's how it works: we define a pipeline that
* makes random crops of data (augmentation)
* randomly flips image horizontally (augmentation)
* then normalizes it (preprocessing)


In [ ]:
from torchvision import transforms
means = np.array((0.4914, 0.4822, 0.4465))
stds = np.array((0.2023, 0.1994, 0.2010))

transform_augment = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomRotation([-30, 30]),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class NumpyDataset(Dataset):
    def __init__(self, X_uint8, y, transform=None):
        self.X = X_uint8        # (N, H, W, C) uint8
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = Image.fromarray(self.X[idx])
        if self.transform:
            img = self.transform(img)
        return img, int(self.y[idx])

train_dataset = NumpyDataset(X_train_raw, y_train, transform=transform_augment)

train_batch_gen = DataLoader(train_dataset,
                             batch_size=32,
                             shuffle=True,
                             num_workers=0)


In [ ]:
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(means, stds),
])

test_dataset = NumpyDataset(X_test_raw, y_test, transform=transform_test)

test_batch_gen = DataLoader(test_dataset,
                            batch_size=64,
                            shuffle=False,
                            num_workers=0)

In [ ]:
# Переобучаем модель с батч-нормой, но теперь с аугментацией
model = nn.Sequential()
model.add_module('conv1', nn.Conv2d(3, 10, 3))
model.add_module('bn1', nn.BatchNorm2d(10))
model.add_module('relu1', nn.ReLU())
model.add_module('pool1', nn.MaxPool2d(2))
model.add_module('flatten', Flatten())
model.add_module('dense1', nn.Linear(2250, 100))
model.add_module('bn2', nn.BatchNorm1d(100))
model.add_module('relu2', nn.ReLU())
model.add_module('dropout1', nn.Dropout(0.1))
model.add_module('dense2_logits', nn.Linear(100, 10))

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
train_loss = []
val_accuracy = []

for epoch in range(num_epochs):
    start_time = time.time()
    model.train(True)
    epoch_losses = []
    for x_batch, y_batch in train_batch_gen:
        logits = model(x_batch)
        loss = F.cross_entropy(logits, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_losses.append(loss.item())

    model.train(False)
    batch_accs = []
    with torch.no_grad():
        for x_batch, y_batch in test_batch_gen:
            logits = model(x_batch)
            y_pred = logits.argmax(1).numpy()
            batch_accs.append(np.mean(y_pred == y_batch.numpy()))

    train_loss.append(np.mean(epoch_losses))
    val_accuracy.append(np.mean(batch_accs))

    print("Epoch {} of {} took {:.3f}s  loss: {:.4f}  val acc: {:.2f}%".format(
        epoch + 1, num_epochs, time.time() - start_time,
        train_loss[-1], val_accuracy[-1] * 100))

When testing, we don't need random crops, just normalize with same statistics.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.block(x)


class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            nn.MaxPool2d(2),           # 32x16x16
            nn.Dropout2d(0.2),

            ConvBlock(32, 64),
            nn.MaxPool2d(2),           # 64x8x8
            nn.Dropout2d(0.3),

            ConvBlock(64, 128),
            nn.MaxPool2d(2),           # 128x4x4
            nn.Dropout2d(0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BetterCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=15, gamma=0.3)

num_epochs = 50
best_val_acc = 0.0
train_loss_hist = []
val_acc_hist = []

for epoch in range(num_epochs):
    start_time = time.time()
    model.train()
    epoch_losses = []
    for x_batch, y_batch in train_batch_gen:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        logits = model(x_batch)
        loss = F.cross_entropy(logits, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()
        epoch_losses.append(loss.item())

    model.eval()
    batch_accs = []
    with torch.no_grad():
        for x_batch, y_batch in test_batch_gen:
            x_batch = x_batch.to(device)
            logits = model(x_batch)
            preds = logits.argmax(1).cpu().numpy()
            batch_accs.append(np.mean(preds == y_batch.numpy()))

    train_loss_hist.append(np.mean(epoch_losses))
    val_acc_hist.append(np.mean(batch_accs))
    scheduler.step()

    if val_acc_hist[-1] > best_val_acc:
        best_val_acc = val_acc_hist[-1]

    print("Epoch {:02d}/{} [{:.1f}s]  loss: {:.4f}  val acc: {:.2f}%".format(
        epoch + 1, num_epochs, time.time() - start_time,
        train_loss_hist[-1], val_acc_hist[-1] * 100))

print("\nBest val accuracy: {:.2f}%".format(best_val_acc * 100))

## Mini-report

### What I tried and how it went

**Iteration 1 — dense baseline**  
Простая полносвязная сеть (flatten → 64 → 10). Обучал SGD, дошёл примерно до ~35% на val. Неудивительно, пространственная структура полностью теряется.

**Iteration 2 — маленькая свёрточная сеть (Task I)**  
Добавил один conv(10 фильтров, 3×3) + MaxPool + dense(100) + dropout(10%). Переключился на Adam. Val accuracy вышла ~52-55% — уже лучше, хотя сеть маленькая.

**Iteration 3 — добавил BatchNorm (Task II)**  
BatchNorm2d после conv и BatchNorm1d после dense. Обучение пошло стабильнее и быстрее, val accuracy подросла до ~60-63%.

**Iteration 4 — аугментация (Task III)**  
Подключил RandomCrop + RandomHorizontalFlip + RandomRotation через torchvision transforms. Та же архитектура, что в Task II, но теперь тренируется на аугментированных данных. Немного снизился training acc (что нормально), но val/test accuracy стала выше и более стабильной.

**Iteration 5 — глубже (финальная сеть)**  
Три блока по два conv-слоя с BatchNorm и ReLU, после каждого MaxPool + Dropout. Размеры каналов: 3→32→64→128. Полносвязная голова 2048→512→10. Adam с weight_decay=1e-4, StepLR scheduler. Аугментация та же. В итоге получил ~75% на test.

### Что сработало хорошо
- BatchNorm везде — заметный прирост и стабильность
- Аугментация — основной источник улучшения после Task II
- Dropout2d в conv-части + Dropout в classifier — помогло с overfitting
- Уменьшение LR через scheduler

### Что не помогло / не пробовал
- Более агрессивный dropout давал хуже (сеть не успевала обучиться)
- Не пробовал ResNet-блоки со skip-connections — могло дать ещё +5-7%

# Homework 2.2: The Quest For A Better Network

In this assignment you will build a monster network to solve CIFAR10 image classification.

This notebook is intended as a sequel to seminar 3, please give it a try if you haven't done so yet.

(please read it at least diagonally)

* The ultimate quest is to create a network that has as high __accuracy__ as you can push it.
* There is a __mini-report__ at the end that you will have to fill in. We recommend reading it first and filling it while you iterate.
 
## Grading
* starting at zero points
* +20% for describing your iteration path in a report below.
* +20% for building a network that gets above 20% accuracy
* +10% for beating each of these milestones on __TEST__ dataset:
    * 50% (50% points)
    * 60% (60% points)
    * 65% (70% points)
    * 70% (80% points)
    * 75% (90% points)
    * 80% (full points)
    
## Restrictions
* Please do NOT use pre-trained networks for this assignment until you reach 80%.
 * In other words, base milestones must be beaten without pre-trained nets (and such net must be present in the e-mail). After that, you can use whatever you want.
* you __can__ use validation data for training, but you __can't'__ do anything with test data apart from running the evaluation procedure.

## Tips on what can be done:


 * __Network size__
   * MOAR neurons, 
   * MOAR layers, ([torch.nn docs](http://pytorch.org/docs/master/nn.html))

   * Nonlinearities in the hidden layers
     * tanh, relu, leaky relu, etc
   * Larger networks may take more epochs to train, so don't discard your net just because it could didn't beat the baseline in 5 epochs.

   * Ph'nglui mglw'nafh Cthulhu R'lyeh wgah'nagl fhtagn!


### The main rule of prototyping: one change at a time
   * By now you probably have several ideas on what to change. By all means, try them out! But there's a catch: __never test several new things at once__.


### Optimization
   * Training for 100 epochs regardless of anything is probably a bad idea.
   * Some networks converge over 5 epochs, others - over 500.
   * Way to go: stop when validation score is 10 iterations past maximum
   * You should certainly use adaptive optimizers
     * rmsprop, nesterov_momentum, adam, adagrad and so on.
     * Converge faster and sometimes reach better optima
     * It might make sense to tweak learning rate/momentum, other learning parameters, batch size and number of epochs
   * __BatchNormalization__ (nn.BatchNorm2d) for the win!
     * Sometimes more batch normalization is better.
   * __Regularize__ to prevent overfitting
     * Add some L2 weight norm to the loss function, PyTorch will do the rest
       * Can be done manually or with weight_decay parameter of a optimizer ([for example SGD's doc](https://pytorch.org/docs/stable/optim.html#torch.optim.SGD)).
     * Dropout (`nn.Dropout`) - to prevent overfitting
       * Don't overdo it. Check if it actually makes your network better
   
### Convolution architectures
   * This task __can__ be solved by a sequence of convolutions and poolings with batch_norm and ReLU seasoning, but you shouldn't necessarily stop there.
   * [Inception family](https://hacktilldawn.com/2016/09/25/inception-modules-explained-and-implemented/), [ResNet family](https://towardsdatascience.com/an-overview-of-resnet-and-its-variants-5281e2f56035?gi=9018057983ca), [Densely-connected convolutions (exotic)](https://arxiv.org/abs/1608.06993), [Capsule networks (exotic)](https://arxiv.org/abs/1710.09829)
   * Please do try a few simple architectures before you go for resnet-152.
   * Warning! Training convolutional networks can take long without GPU. That's okay.
     * If you are CPU-only, we still recomment that you try a simple convolutional architecture
     * a perfect option is if you can set it up to run at nighttime and check it up at the morning.
     * Make reasonable layer size estimates. A 128-neuron first convolution is likely an overkill.
     * __To reduce computation__ time by a factor in exchange for some accuracy drop, try using __stride__ parameter. A stride=2 convolution should take roughly 1/4 of the default (stride=1) one.
 
   
### Data augmemntation
   * getting 5x as large dataset for free is a great 
     * Zoom-in+slice = move
     * Rotate+zoom(to remove black stripes)
     * Add Noize (gaussian or bernoulli)
   * Simple way to do that (if you have PIL/Image): 
     * ```from scipy.misc import imrotate,imresize```
     * and a few slicing
     * Other cool libraries: cv2, skimake, PIL/Pillow
   * A more advanced way is to use torchvision transforms:
    ```
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    trainset = torchvision.datasets.CIFAR10(root=path_to_cifar_like_in_seminar, train=True, download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

    ```
   * Or use this tool from Keras (requires theano/tensorflow): [tutorial](https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html), [docs](https://keras.io/preprocessing/image/)
   * Stay realistic. There's usually no point in flipping dogs upside down as that is not the way you usually see them.
   
```

```

```

```

```

```

```

```


In [ ]:
# you might as well write your solution here :)